In [3]:
import pandas as pd
df=pd.read_csv('top100.csv')
df=df[df['rank']==1]
df

,query_index,method,rank,PaperID,similarity
0,0,raw,1,2533304691,0.661030
100,1,raw,1,2343633027,0.637669
200,2,raw,1,2232906764,0.785529
300,3,raw,1,2568152049,0.705435
400,4,raw,1,2473167446,0.724842
...,...,...,...,...,...
899500,1495,one_shot,1,2518742280,0.530622
899600,1496,one_shot,1,2484695545,0.628557
899700,1497,one_shot,1,2546425461,0.548855
899800,1498,one_shot,1,2569917959,0.666376


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
def mean_ci(series, confidence=0.95):
    data = np.array(series)
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)  # standard error
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return pd.Series({"mean": mean})
# Group by method and calculate mean & CI for "similarity"
result = df.groupby("method")["similarity"].apply(mean_ci).reset_index()
print(result)

     method level_1  similarity
0       cot    mean    0.635211
1       got    mean    0.637814
2  one_shot    mean    0.632377
3       raw    mean    0.639301
4       tot    mean    0.635824


In [ ]:
import pandas as pd
df=pd.read_parquet('/net/scratch/honglinbao/coh_data-2016.parquet')
import re
df = df[df['recovered_abstract'].apply(lambda x: bool(re.match(r'^[\x00-\x7F]+$', str(x))))]
df.reset_index(drop=True, inplace=True)
df

,PaperID,FieldOfStudyName,recovered_abstract
0,2203024088,developmental psychology,posttraumatic stress disorder (ptsd) is charac...
1,2954553731,arc,a nasal dilator and method of making nasal dil...
2,2746546208,point,a system and method for method for identifying...
3,2881689944,calibration,the invention discloses a worm-drive simple so...
4,2841760640,point,the invention discloses a condensation water p...
...,...,...,...
1057582,2991518598,blood pressure,diseases of blood vessels cause more morbidity...
1057583,2494930754,component,architecture description languages (adls) enca...
1057584,2867964721,information processing,problem to be solved: to reduce a waiting time...
1057585,2520247297,menstruation,"hypermobile ehlers-danlos syndrome (heds), is ..."


In [3]:
df2=df.sample(1000)
df2.to_csv('sample.csv')

In [4]:
df.to_parquet('/net/scratch/honglinbao/coh_data-2016.parquet',index=False)

In [ ]:
import pandas as pd
from openai import AzureOpenAI

# Azure OpenAI configuration
endpoint = "https://hypoextraction.cognitiveservices.azure.com/"
model_name = "o3-mini"
deployment = "o3-mini"
subscription_key = "9E35cCPh8IS7Ulw0KYB3y2Ehl3vTv40TIVORaBKEMzJ9fa1N6BfgJQQJ99BFACYeBjFXJ3w3AAAAACOGtwQl"
api_version = "2024-12-01-preview"

# Initialize client
client = AzureOpenAI(
    api_key=subscription_key,
    api_version=api_version,
    azure_endpoint=endpoint
)

# Function to rewrite text
def rewrite(text):
    try:
        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": "You are a helpful writer."},
                {"role": "user", "content": f"Rewrite the following text but keep the original meaning:\n\n{text}"}
            ],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

# Read CSV
df = pd.read_csv('sample.csv')
df=df.head(10)
# Apply rewrite to each row in 'raw'
df["rewritten"] = df["raw"].apply(rewrite)
df.to_csv('sample.csv',index=False)

,raw,rewritten
0,the invention relates to a system and a method...,The invention relates to a system and method f...
1,"epure m., prior d. and serarols c. assessing t...","Epure M., Prior D., and Serarols C. take a reg..."
2,urea sensor seal structure that is qualified f...,"The urea sensor’s sealing structure, which mee..."
3,the utility model provides a novel obviously a...,The utility model discloses a novel air purifi...
4,the invention provides an album-based social g...,The invention describes a method and device fo...
5,"in this paper, an intelligent scheme for detec...",This paper introduces an intelligent approach ...
6,cholangiocarcinoma (cca) is a heterogeneous gr...,Cholangiocarcinoma (CCA) comprises a diverse g...
7,the utility model discloses an ore material bl...,The utility model discloses an ore material bl...
8,the emergence of social enterprises (ses) with...,"In recent years, there has been a significant ..."
9,the invention relates to a laser engraving man...,The invention pertains to a laser engraving pr...


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
df = pd.read_csv('sample.csv')

# Load the all-mpnet-base-v2 model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Compute embeddings for raw and rewritten columns
embeddings_raw = model.encode(df['raw'].tolist(), convert_to_tensor=True)
embeddings_rewritten = model.encode(df['rewritten'].tolist(), convert_to_tensor=True)

# Calculate cosine similarity
similarities = util.cos_sim(embeddings_raw, embeddings_rewritten).diagonal()

# Add similarity column to DataFrame
df['similarity'] = similarities.cpu().numpy()
df.to_csv('sample.csv',index=False)

In [3]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the CSV
df = pd.read_csv('sample.csv')

# Calculate mean
mean_similarity = df['similarity'].mean()

# 95% Confidence Interval (CI) for the mean
confidence_level = 0.95
n = len(df['similarity'])
std_err = stats.sem(df['similarity'], nan_policy='omit')
h = std_err * stats.t.ppf((1 + confidence_level) / 2, n - 1)

ci_lower = mean_similarity - h
ci_upper = mean_similarity + h
mean_similarity, ci_lower, ci_upper

(0.9415791811011011, 0.9382292339696992, 0.944929128232503)

0       asthma exacerbation during pregnancy poses sig...
1       managing chronic medical conditions during pre...
2       asthma is a chronic medical condition that com...
3       the integration of electric vehicles (evs) int...
4       asthma exacerbations during pregnancy pose sig...
                              ...                        
2995    this paper presents a novel approach to poweri...
2996    this paper presents a novel interdisciplinary ...
2997    this study combines insights from humanities a...
2998    the integration of materials science and cultu...
2999    this interdisciplinary research project integr...
Name: raw, Length: 3000, dtype: object

In [7]:
import pandas as pd
all = pd.read_parquet('/net/scratch/honglinbao/coh_data-2016.parquet')
all

,PaperID,FieldOfStudyName,recovered_abstract
0,2203024088,developmental psychology,posttraumatic stress disorder (ptsd) is charac...
1,2954553731,arc,a nasal dilator and method of making nasal dil...
2,2746546208,point,a system and method for method for identifying...
3,2881689944,calibration,the invention discloses a worm-drive simple so...
4,2841760640,point,the invention discloses a condensation water p...
...,...,...,...
1057582,2991518598,blood pressure,diseases of blood vessels cause more morbidity...
1057583,2494930754,component,architecture description languages (adls) enca...
1057584,2867964721,information processing,problem to be solved: to reduce a waiting time...
1057585,2520247297,menstruation,"hypermobile ehlers-danlos syndrome (heds), is ..."


In [9]:
import pandas as pd
df = pd.read_csv('/home/honglinbao/chain_of_hints/diff.csv')
id_pairs_dup = df.duplicated(subset=['id1', 'id2']) | df.duplicated(subset=['id2', 'id1'])
paper_pairs_dup = df.duplicated(subset=['paper1', 'paper2']) | df.duplicated(subset=['paper2', 'paper1'])
abstract_pairs_dup = df.duplicated(subset=['abstract1', 'abstract2']) | df.duplicated(subset=['abstract2', 'abstract1'])
fieldpair_stats = df['FieldPair'].value_counts().reset_index()
fieldpair_stats.columns = ['FieldPair', 'count']
fieldpair_stats['proportion'] = fieldpair_stats['count'] / len(df)
min_words_abs1 = df['abstract1'].str.split().apply(len).min()
min_words_abs2 = df['abstract2'].str.split().apply(len).min()
min_words_abs2

46

In [ ]:
num_unique_duplicates = pairs[duplicate_mask].nunique()
unique_duplicate_pairs = pairs[duplicate_mask].unique()

print("唯一重复组合数:", num_unique_duplicates)
print("示例重复组合:", unique_duplicate_pairs[:10])

唯一重复组合数: 558
示例重复组合: [frozenset({'poster presentation background asthma is one of the leading chronic medical problems exacerbated during pregnancy, and this condition affects approximately 3% to 8% of the pregnant population. the health care team needs to be educated on factors that may trigger an exacerbation of asthma in pregnancy, appropriate management, treatment techniques to share with their patients, and the complications that an exacerbation of maternal asthma may pass on to the newborn. the goal for all asthmatic patients is to remain asymptomatic during pregnancy to assist in improving maternal and newborn outcomes. case a 30‐year‐old, g4p1 woman at 27 weeks gestation was transported by air to our large level‐iii mid atlantic trauma center from a satellite medical emergency department with status asthmaticus for further management. presenting with possible influenza virus, her history included asthma, positive cigarette use, and a recent respiratory infection. her respirator

In [6]:
import pandas as pd
df = pd.read_csv('top-matches_190.csv')
#df = df[df['rank'] ==1]
avg_similarity = df.groupby('method')['similarity'].mean()
print(avg_similarity)

method
cot         0.526993
few_shot    0.524986
gen_abs     0.523539
got         0.527043
one_shot    0.534449
raw         0.529710
tot         0.534436
Name: similarity, dtype: float64
